<a href="https://colab.research.google.com/github/DrLuisZapata/lidar-cnn-lstm-curves/blob/main/CNN_LSTM_PIDL_PandaSet_KITTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physics-Informed CNN-LSTM for Speed and Heading-Rate Prediction from LiDAR Road Geometry

Companion notebook of the article *"A Physics-Informed CNN-LSTM for Speed and Heading-Rate Prediction from LiDAR Road Geometry toward Safe Curve Negotiation"*.

**Pipeline**

1. Per-frame features computed **in the same way from LiDAR in both datasets** (PandaSet and KITTI raw, Road category): lateral offset of the road ahead at several look-ahead distances (road-geometry preview), banking angle and grade, plus vehicle speed and heading rate.
2. Sliding windows of 2.0 s; targets are the mean heading rate and mean speed over a horizon of **1, 2 and 3 s**.
3. Multi-task CNN-LSTM that predicts the **change** with respect to the last observed values, trained with a one-sided physics-informed penalty based on the maximum cornering speed of a banked curve.
4. Sequence-wise train / validation / test split on PandaSet with early stopping on the validation sequences; zero-shot evaluation on KITTI.
5. Comparison with a persistence baseline and with the same network without the physics term (ablation).

## 0. Kaggle credentials (optional)

Only needed if PandaSet is downloaded through the Kaggle API. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab **Secrets** panel first.

In [ ]:
# Configure Kaggle API credentials from Colab Secrets.
# The credentials are written to ~/.kaggle/kaggle.json (outside the notebook),
# so they are never stored in the notebook itself.

import json, os
from google.colab import userdata

# Read username and API key from the Colab Secrets panel
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Create the Kaggle configuration file with restricted permissions (owner read/write only)
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print("Kaggle credentials configured securely (not stored in the notebook).")

## 1. Google Drive and global parameters

In [ ]:
# Mount Google Drive (PandaSet sequences are expected under /content/drive/MyDrive/pandaset/).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Global parameters of the experiment.

import os, glob, json
import numpy as np
import pandas as pd

FRECUENCIA_HZ = 10                      # PandaSet and KITTI raw (synced) are sampled at 10 Hz
WINDOW_SIZE = 20                        # 20 past frames = 2.0 s of history
HORIZONTES = [10, 20, 30]               # prediction horizons: 1.0, 2.0 and 3.0 s
HORIZONTE_PRINCIPAL = 10                # horizon used for the main tables and figures (1.0 s)
DISTANCIAS_PREVIEW = [10, 20, 30, 40]   # look-ahead distances (m) for the road-geometry preview
SEMILLA = 42
N_PARTICIONES = 5                       # number of different sequence-wise splits (repetitions)

RUTA_PANDASET = "/content/drive/MyDrive/pandaset/"
RUTA_KITTI = "/content/kitti_road/"

# Network inputs (same definition in PandaSet and KITTI)
FEATURE_COLS = [f"preview_{d}m" for d in DISTANCIAS_PREVIEW] + ["peralte_deg", "pendiente_pct", "velocidad_kmh", "heading_rate"]
N_FEATURES = len(FEATURE_COLS)
IDX_VEL, IDX_HR = FEATURE_COLS.index("velocidad_kmh"), FEATURE_COLS.index("heading_rate")

# Physical constants of the physics-informed term
G_FISICA = 9.81      # m/s^2
MU_FISICA = 0.7      # conservative dry-pavement friction coefficient
LAMBDA_FISICA = 0.1  # weight of the physics penalty

print(f"Features ({N_FEATURES}): {FEATURE_COLS}")
print(f"Window: {WINDOW_SIZE / FRECUENCIA_HZ:.1f} s | horizons: {[h / FRECUENCIA_HZ for h in HORIZONTES]} s")

## 2. Geometry utilities

- `calcular_yaw_trayectoria`: vehicle yaw from the sensor trajectory (backward difference, unwrapped, held while stopped).
- `mundo_a_vehiculo`: transforms PandaSet points from world coordinates to a vehicle-aligned frame (`x` forward, `y` left, `z` vertical). KITTI Velodyne points are already in the sensor frame (`x` forward, `y` left, `z` up).
- `extraer_features_lidar`: road-geometry features from one LiDAR frame in the vehicle frame, identical for both datasets.
- `calcular_vmax_fisico_kmh`: maximum cornering speed of a banked curve with friction.

In [ ]:
# Geometry utilities shared by the PandaSet and KITTI pipelines.

MIN_DESPLAZAMIENTO_M = 0.05   # below this displacement per frame (0.5 m/s) the yaw is held constant

def calcular_yaw_trayectoria(x, y, min_desp=MIN_DESPLAZAMIENTO_M):
    """Yaw (rad) per frame from the trajectory: backward difference, held while stopped, unwrapped."""
    n = len(x)
    yaw = np.full(n, np.nan)
    ultimo = None
    for k in range(1, n):
        dx, dy = x[k] - x[k - 1], y[k] - y[k - 1]
        if np.hypot(dx, dy) >= min_desp:
            ultimo = np.arctan2(dy, dx)
        if ultimo is not None:
            yaw[k] = ultimo
    validos = ~np.isnan(yaw)
    if not validos.any():
        return np.zeros(n)
    yaw[~validos] = yaw[validos][0]
    return np.unwrap(yaw)


def mundo_a_vehiculo(puntos, origen_xy, yaw):
    """World -> vehicle-aligned frame: translate to the sensor (x, y) and rotate by -yaw; z unchanged."""
    c, s = np.cos(yaw), np.sin(yaw)
    dx = puntos[:, 0] - origen_xy[0]
    dy = puntos[:, 1] - origen_xy[1]
    return np.column_stack([c * dx + s * dy, -s * dx + c * dy, puntos[:, 2]])


def extraer_features_lidar(p):
    """
    Road-geometry features from one LiDAR frame in the vehicle frame (x forward, y left, z up).

    Returns a dict with:
        preview_<d>m: lateral offset (m) of the center of the ground extent in a 5 m slice
                      centered at d meters ahead (NaN if the slice has too few ground points)
        peralte_deg:  banking angle (deg) between the right and left halves of the ground ahead
        pendiente_pct: road grade (%) between near (<10 m) and far (>40 m) ground points
        theta_rad:    banking angle in radians, used only in the physics-informed term
    """
    salida = {f"preview_{d}m": np.nan for d in DISTANCIAS_PREVIEW}
    salida.update({"peralte_deg": np.nan, "pendiente_pct": np.nan, "theta_rad": np.nan})

    # Region of interest: 0-50 m ahead, +/- 15 m laterally
    roi = p[(p[:, 0] > 0) & (p[:, 0] < 50) & (np.abs(p[:, 1]) < 15)]
    if len(roi) < 50:
        return salida

    # Ground points: band around the 10th percentile of height in the region of interest
    z_base = np.percentile(roi[:, 2], 10)
    suelo = roi[(roi[:, 2] > z_base - 0.15) & (roi[:, 2] < z_base + 0.30)]
    if len(suelo) < 50:
        return salida

    # Road-geometry preview: center of the ground extent at each look-ahead distance
    for d in DISTANCIAS_PREVIEW:
        tramo = suelo[np.abs(suelo[:, 0] - d) < 2.5]
        if len(tramo) >= 10:
            salida[f"preview_{d}m"] = 0.5 * (np.percentile(tramo[:, 1], 10) + np.percentile(tramo[:, 1], 90))

    # Grade: height change between far (>40 m) and near (<10 m) ground points over ~30 m
    lejos, cerca = suelo[suelo[:, 0] > 40, 2], suelo[suelo[:, 0] < 10, 2]
    if len(lejos) > 0 and len(cerca) > 0:
        salida["pendiente_pct"] = (lejos.mean() - cerca.mean()) / 30 * 100

    # Banking: right (y below the near-road center) vs. left (y above), within 20 m ahead
    cercano = suelo[suelo[:, 0] < 20]
    if len(cercano) >= 20:
        centro = np.median(cercano[:, 1])
        der, izq = cercano[cercano[:, 1] < centro], cercano[cercano[:, 1] > centro]
        if len(der) > 0 and len(izq) > 0:
            rise = der[:, 2].mean() - izq[:, 2].mean()
            run = max(abs(der[:, 1].mean() - izq[:, 1].mean()), 0.5)
            theta = np.arctan2(rise, run)
            salida["theta_rad"] = theta
            salida["peralte_deg"] = np.degrees(theta)
    return salida


def calcular_vmax_fisico_kmh(R, theta, g=G_FISICA, mu=MU_FISICA):
    """v_max = sqrt(g*R*(tan(theta)+mu) / (1-mu*tan(theta))), in km/h."""
    tan_theta = np.tan(theta)
    denom = np.clip(1.0 - mu * tan_theta, 0.05, None)
    return np.sqrt(np.clip(g * R * (tan_theta + mu) / denom, 0.0, None)) * 3.6


def completar_dinamica(df):
    """
    Fill missing geometry values (forward fill, then 0), and add the physical turning radius
    R = v / omega (clipped to 5-1000 m) and the physical speed limit v_max for each frame.
    """
    geo = [f"preview_{d}m" for d in DISTANCIAS_PREVIEW] + ["peralte_deg", "pendiente_pct", "theta_rad"]
    df[geo] = df[geo].ffill().fillna(0.0)
    omega = np.maximum(np.abs(df["heading_rate"].values) * FRECUENCIA_HZ, 1e-3)   # rad/s
    df["R_m"] = np.clip((df["velocidad_kmh"].values / 3.6) / omega, 5.0, 1000.0)
    df["vmax_kmh"] = calcular_vmax_fisico_kmh(df["R_m"].values, df["theta_rad"].values)
    return df

## 3. PandaSet: per-frame features

PandaSet stores LiDAR points in world coordinates; each frame is transformed to the vehicle frame with the yaw estimated from the LiDAR sensor trajectory. Speed and heading rate come from the same sensor poses.

In [ ]:
# Per-frame features for one PandaSet sequence.

def cargar_secuencia_pandaset(id_secuencia, ruta_base=RUTA_PANDASET):
    ruta_lidar = os.path.join(ruta_base, id_secuencia, "lidar")
    ruta_poses = os.path.join(ruta_lidar, "poses.json")
    if not os.path.exists(ruta_poses):
        return None
    archivos = sorted(f for f in os.listdir(ruta_lidar) if f.endswith(".pkl"))
    with open(ruta_poses, "r") as f:
        poses = json.load(f)
    n = min(len(archivos), len(poses))
    if n < 2:
        return None

    px = np.array([poses[i]["position"]["x"] for i in range(n)])
    py = np.array([poses[i]["position"]["y"] for i in range(n)])
    pz = np.array([poses[i]["position"]["z"] for i in range(n)])

    # Speed (km/h) from the 3D displacement of the sensor between consecutive frames
    desp = np.sqrt(np.diff(px)**2 + np.diff(py)**2 + np.diff(pz)**2)
    velocidad = np.concatenate([[desp[0]], desp]) * FRECUENCIA_HZ * 3.6

    # Yaw and heading rate (rad/frame, trailing 3-frame moving average, no look-ahead)
    yaw = calcular_yaw_trayectoria(px, py)
    heading_rate = pd.Series(np.diff(yaw, prepend=yaw[0])).rolling(3, min_periods=1).mean().values

    filas = []
    for i in range(n):
        try:
            puntos = pd.read_pickle(os.path.join(ruta_lidar, archivos[i]))[["x", "y", "z"]].values
            feats = extraer_features_lidar(mundo_a_vehiculo(puntos, (px[i], py[i]), yaw[i]))
        except Exception:
            feats = extraer_features_lidar(np.zeros((0, 3)))
        feats.update({"frame": i, "velocidad_kmh": velocidad[i], "heading_rate": heading_rate[i]})
        filas.append(feats)
    return completar_dinamica(pd.DataFrame(filas))

## 4. KITTI raw (Road category): download and per-frame features

KITTI is downloaded from its official AWS mirror (no login required). The road-geometry features are computed from the **Velodyne point clouds** with the same function used for PandaSet; speed and heading rate come from the OXTS GPS/IMU.

In [ ]:
# Download and extract the selected KITTI raw sequences (synced data + calibration).

import subprocess, zipfile

# Official "Road" category sequences of KITTI raw (date, drive)
SECUENCIAS_KITTI_ROAD = [
    ("2011_09_26", "0015"),
    ("2011_09_26", "0027"),
    ("2011_09_26", "0028"),
]

os.makedirs(RUTA_KITTI, exist_ok=True)

def zip_valido(ruta_zip):
    """Check that the file exists and is a valid .zip (not an empty/corrupt file from a failed download)."""
    if not os.path.exists(ruta_zip) or os.path.getsize(ruta_zip) < 1024:
        return False
    return zipfile.is_zipfile(ruta_zip)

def descargar_secuencia_kitti(fecha, drive_id):
    """Download (if needed) and extract one KITTI sequence and its calibration files; return the sequence folder."""
    base_url = "https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data"
    nombre_sync = f"{fecha}_drive_{drive_id}_sync"
    zip_sync = f"{nombre_sync}.zip"
    zip_calib = f"{fecha}_calib.zip"

    destino = os.path.join(RUTA_KITTI, fecha)
    os.makedirs(destino, exist_ok=True)

    for zip_name, url in [
        (zip_calib, f"{base_url}/{zip_calib}"),
        (zip_sync, f"{base_url}/{fecha}_drive_{drive_id}/{zip_sync}"),
    ]:
        ruta_zip = os.path.join(destino, zip_name)
        # Download only if the zip is missing or corrupt
        if not zip_valido(ruta_zip):
            if os.path.exists(ruta_zip):
                os.remove(ruta_zip)  # remove a previous corrupt/incomplete download
            print(f"Downloading {zip_name} ...")
            subprocess.run(["wget", "-q", "-O", ruta_zip, url], check=True)
            if not zip_valido(ruta_zip):
                raise ValueError(f"The downloaded file {zip_name} is not a valid zip.")
        subprocess.run(["unzip", "-oq", ruta_zip, "-d", destino], check=True)

    # The KITTI zip already contains a folder named after the date,
    # so the extracted data end up one level deeper than 'destino'.
    return os.path.join(destino, fecha, nombre_sync)

# Download all selected sequences and keep the paths of those that succeed
rutas_secuencias_kitti = []
for fecha, drive_id in SECUENCIAS_KITTI_ROAD:
    try:
        ruta = descargar_secuencia_kitti(fecha, drive_id)
        assert os.path.isdir(ruta), f"Expected path not found: {ruta}"
        rutas_secuencias_kitti.append(ruta)
        print(f"OK: {ruta}")
    except Exception as e:
        print(f"Error with {fecha}_drive_{drive_id}: {e}")

In [ ]:
# Per-frame features for one KITTI sequence (Velodyne LiDAR + OXTS).

OXTS_COLS = [
    "lat", "lon", "alt", "roll", "pitch", "yaw", "vn", "ve", "vf", "vl", "vu",
    "ax", "ay", "az", "af", "al", "au", "wx", "wy", "wz", "wf", "wl", "wu",
    "pos_accuracy", "vel_accuracy", "navstat", "numsats", "posmode", "velmode", "orimode",
]

def cargar_secuencia_kitti(ruta_secuencia):
    ruta_oxts = os.path.join(ruta_secuencia, "oxts", "data")
    ruta_velo = os.path.join(ruta_secuencia, "velodyne_points", "data")
    if not os.path.isdir(ruta_oxts) or not os.path.isdir(ruta_velo):
        print(f"Missing oxts or velodyne data in {ruta_secuencia}")
        return None
    archivos_oxts = sorted(glob.glob(os.path.join(ruta_oxts, "*.txt")))
    archivos_velo = sorted(glob.glob(os.path.join(ruta_velo, "*.bin")))
    n = min(len(archivos_oxts), len(archivos_velo))

    oxts = pd.DataFrame([np.loadtxt(a) for a in archivos_oxts[:n]], columns=OXTS_COLS)
    yaw = np.unwrap(oxts["yaw"].values)
    heading_rate = pd.Series(np.diff(yaw, prepend=yaw[0])).rolling(3, min_periods=1).mean().values

    filas = []
    for i in range(n):
        # Velodyne frame: x forward, y left, z up (float32 x, y, z, reflectance)
        puntos = np.fromfile(archivos_velo[i], dtype=np.float32).reshape(-1, 4)[:, :3]
        feats = extraer_features_lidar(puntos.astype(np.float64))
        feats.update({"frame": i, "velocidad_kmh": oxts["vf"].iloc[i] * 3.6, "heading_rate": heading_rate[i]})
        filas.append(feats)
    return completar_dinamica(pd.DataFrame(filas))

## 5. Extract features for all sequences

In [ ]:
# Extract per-frame features once for every PandaSet and KITTI sequence (slow step: reads all LiDAR frames).

dfs_pandaset = {}
for seq in sorted(d for d in os.listdir(RUTA_PANDASET) if os.path.isdir(os.path.join(RUTA_PANDASET, d))):
    df_seq = cargar_secuencia_pandaset(seq)
    if df_seq is not None and len(df_seq) >= WINDOW_SIZE + max(HORIZONTES) + 1:
        dfs_pandaset[seq] = df_seq
print(f"PandaSet sequences with enough frames: {len(dfs_pandaset)}")

dfs_kitti = {}
for ruta in rutas_secuencias_kitti:
    df_seq = cargar_secuencia_kitti(ruta)
    if df_seq is not None:
        dfs_kitti[os.path.basename(ruta)] = df_seq
print(f"KITTI sequences: {list(dfs_kitti)}")

# Quick check of the feature ranges in both datasets (they should be comparable now)
resumen_feats = pd.DataFrame({
    "PandaSet mean": pd.concat(dfs_pandaset.values())[FEATURE_COLS].mean(),
    "PandaSet std": pd.concat(dfs_pandaset.values())[FEATURE_COLS].std(),
    "KITTI mean": pd.concat(dfs_kitti.values())[FEATURE_COLS].mean(),
    "KITTI std": pd.concat(dfs_kitti.values())[FEATURE_COLS].std(),
})
display(resumen_feats.round(3))

## 6. Sequences and sequence-wise splits

Each sample has a 20-frame input window. The targets are the mean heading rate (deg/frame) and mean speed (km/h) over the next `horizon` frames. The physics term uses the most restrictive physical limit (minimum `v_max`) within the horizon.

To check that the results do not depend on which sequences end up in the test set, the whole experiment is repeated with `N_PARTICIONES` different random splits of the PandaSet sequences into training (about 68%), validation (about 12%) and test (20%). Within each repetition, the same split is used for all horizons.

In [ ]:
# Sliding-window samples and the fixed sequence-wise split.

def construir_secuencias(df, horizon):
    datos = df[FEATURE_COLS].values
    giro = np.degrees(df["heading_rate"].values)
    vel = df["velocidad_kmh"].values
    vmax = df["vmax_kmh"].values
    X, yg, yv, vm = [], [], [], []
    for i in range(len(df) - WINDOW_SIZE - horizon + 1):
        fut = slice(i + WINDOW_SIZE, i + WINDOW_SIZE + horizon)
        X.append(datos[i:i + WINDOW_SIZE])
        yg.append(giro[fut].mean())
        yv.append(vel[fut].mean())
        vm.append(vmax[fut].min())
    return np.array(X), np.array(yg), np.array(yv), np.array(vm)


def apilar(dfs, ids, horizon):
    """Concatenate the samples of several sequences."""
    partes = [construir_secuencias(dfs[s], horizon) for s in ids]
    return [np.concatenate([p[k] for p in partes]) for k in range(4)]


def crear_particion(semilla):
    """Random sequence-wise split of the PandaSet sequences into train / validation / test."""
    rng = np.random.default_rng(semilla)
    ids = np.array(sorted(dfs_pandaset))
    rng.shuffle(ids)
    n_test = int(round(0.20 * len(ids)))
    n_val = int(round(0.15 * (len(ids) - n_test)))
    return ids[n_test + n_val:], ids[n_test:n_test + n_val], ids[:n_test]

ids_kitti = sorted(dfs_kitti)
for r in range(N_PARTICIONES):
    tr, va, te = crear_particion(SEMILLA + r)
    print(f"Split {r}: train {len(tr)}, validation {len(va)}, test {len(te)} sequences | test: {list(te)}")

## 7. Model

Smaller CNN-LSTM with dropout to reduce overfitting. The network predicts the **standardized change** of heading rate and speed with respect to the last observed frame. The physics layer penalizes `last speed + predicted change > v_max`, expressed in the same standardized space.

In [ ]:
# CNN-LSTM with a physics-informed loss layer.

import keras
from tensorflow.keras import layers, models, Input


class PerdidaFisica(layers.Layer):
    """Adds lam * mean(ReLU(pred - limit)^2) to the training loss and returns pred unchanged."""
    def __init__(self, lam, **kwargs):
        super().__init__(**kwargs)
        self.lam = lam

    def call(self, inputs):
        pred, limite = inputs
        self.add_loss(self.lam * keras.ops.mean(keras.ops.square(keras.ops.relu(pred - limite))))
        return pred


def construir_modelo(lam):
    entradas = Input(shape=(WINDOW_SIZE, N_FEATURES), name="secuencia_entrada")
    limite = Input(shape=(1,), name="vmax_std_input")   # physical limit in the standardized change space

    x = layers.Conv1D(32, 3, padding="causal", activation="relu")(entradas)
    x = layers.Conv1D(16, 3, padding="causal", activation="relu")(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    salida_giro = layers.Dense(1, name="out_giro")(x)
    salida_vel_raw = layers.Dense(1, name="out_velocidad_raw")(x)
    salida_vel = PerdidaFisica(lam, name="out_velocidad")([salida_vel_raw, limite])

    modelo = models.Model([entradas, limite], [salida_giro, salida_vel])
    modelo.compile(optimizer="adam",
                   loss={"out_giro": "mse", "out_velocidad": "mse"},
                   loss_weights={"out_giro": 1.0, "out_velocidad": 1.0})
    return modelo

construir_modelo(LAMBDA_FISICA).summary()

## 8. Experiment for each horizon

For each horizon: build the samples, fit the scalers on the training sequences only, train the CNN-LSTM with and without the physics term (early stopping on the validation sequences), and evaluate the persistence baseline and both networks on the PandaSet test sequences and on KITTI (zero-shot, same scalers).

In [ ]:
# One complete experiment per horizon.

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

def ultimos(X):
    """Last observed heading rate (deg/frame) and speed (km/h) of each window."""
    return np.degrees(X[:, -1, IDX_HR]), X[:, -1, IDX_VEL]


def ejecutar_experimento(h, ids_train, ids_val, ids_test, semilla, verbose=0):
    Xtr, gtr, vtr, mtr = apilar(dfs_pandaset, ids_train, h)
    Xva, gva, vva, mva = apilar(dfs_pandaset, ids_val, h)
    Xte, gte, vte, mte = apilar(dfs_pandaset, ids_test, h)
    Xki, gki, vki, mki = apilar(dfs_kitti, ids_kitti, h)

    # Scalers fitted on the training sequences only
    sx = StandardScaler().fit(Xtr.reshape(-1, N_FEATURES))
    escalar_X = lambda X: sx.transform(X.reshape(-1, N_FEATURES)).reshape(X.shape)
    ug_tr, uv_tr = ultimos(Xtr)
    sg = StandardScaler().fit((gtr - ug_tr).reshape(-1, 1))   # heading-rate change
    sv = StandardScaler().fit((vtr - uv_tr).reshape(-1, 1))   # speed change

    def entradas(X, vmax):
        _, uv = ultimos(X)
        return {"secuencia_entrada": escalar_X(X),
                "vmax_std_input": sv.transform((vmax - uv).reshape(-1, 1))}

    def objetivos(X, g, v):
        ug, uv = ultimos(X)
        return {"out_giro": sg.transform((g - ug).reshape(-1, 1)).ravel(),
                "out_velocidad": sv.transform((v - uv).reshape(-1, 1)).ravel()}

    def predecir(modelo, X, vmax):
        pg, pv = modelo.predict(entradas(X, vmax), verbose=0)
        ug, uv = ultimos(X)
        return ug + sg.inverse_transform(pg).ravel(), uv + sv.inverse_transform(pv).ravel()

    resultados, predicciones, historiales = [], {}, {}
    conjuntos = {"PandaSet": (Xte, gte, vte, mte), "KITTI": (Xki, gki, vki, mki)}

    # Persistence baseline
    for ds, (X, g, v, m) in conjuntos.items():
        pg, pv = ultimos(X)
        predicciones[("Persistence baseline", ds)] = (pg, pv)

    # CNN-LSTM without and with the physics term
    for nombre, lam in [("CNN-LSTM (no PIDL)", 0.0), ("CNN-LSTM + PIDL", LAMBDA_FISICA)]:
        keras.utils.set_random_seed(semilla)
        modelo = construir_modelo(lam)
        parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        hist = modelo.fit(entradas(Xtr, mtr), objetivos(Xtr, gtr, vtr),
                          validation_data=(entradas(Xva, mva), objetivos(Xva, gva, vva)),
                          epochs=100, batch_size=64, callbacks=[parada], verbose=verbose)
        historiales[nombre] = hist.history
        for ds, (X, g, v, m) in conjuntos.items():
            predicciones[(nombre, ds)] = predecir(modelo, X, m)

    for (nombre, ds), (pg, pv) in predicciones.items():
        X, g, v, m = conjuntos[ds]
        resultados.append({
            "Split": semilla - SEMILLA, "Horizon (s)": h / FRECUENCIA_HZ, "Dataset": ds, "Model": nombre,
            "R2 heading rate": r2_score(g, pg), "MSE heading rate": mean_squared_error(g, pg),
            "R2 speed": r2_score(v, pv), "MSE speed": mean_squared_error(v, pv),
            "% above v_max": float(np.mean(pv > m) * 100), "N samples": len(g),
        })
    datos = {"conjuntos": conjuntos, "predicciones": predicciones, "historiales": historiales}
    return pd.DataFrame(resultados), datos


tablas, datos_por_h = [], {}
for r in range(N_PARTICIONES):
    ids_train, ids_val, ids_test = crear_particion(SEMILLA + r)
    for h in HORIZONTES:
        print(f"Split {r + 1}/{N_PARTICIONES}, horizon {h / FRECUENCIA_HZ:.1f} s ...")
        tabla_h, datos = ejecutar_experimento(h, ids_train, ids_val, ids_test, SEMILLA + r)
        tablas.append(tabla_h)
        if r == 0:
            datos_por_h[h] = datos   # predictions of the first split are used for figures 6-8
resultados = pd.concat(tablas, ignore_index=True)
print("Done.")

## 9. Results tables

In [ ]:
# Results: mean and standard deviation over the splits, and paired comparison with persistence.
pd.set_option("display.width", 250)
metricas = ["R2 heading rate", "MSE heading rate", "R2 speed", "MSE speed", "% above v_max"]
claves = ["Horizon (s)", "Dataset", "Model"]

resumen = resultados.groupby(claves)[metricas].agg(["mean", "std"])
display(resumen.round(4))

# Paired comparison per split: relative MSE change of each network with respect to persistence
# (negative = the network has lower error) and number of splits in which the network wins.
base = resultados[resultados["Model"] == "Persistence baseline"].set_index(["Split", "Horizon (s)", "Dataset"])
filas = []
for mod in ["CNN-LSTM (no PIDL)", "CNN-LSTM + PIDL"]:
    red = resultados[resultados["Model"] == mod].set_index(["Split", "Horizon (s)", "Dataset"])
    for salida in ["heading rate", "speed"]:
        cambio = (red[f"MSE {salida}"] - base[f"MSE {salida}"]) / base[f"MSE {salida}"] * 100
        gana = (red[f"MSE {salida}"] < base[f"MSE {salida}"])
        tabla = pd.DataFrame({"change": cambio, "wins": gana}).groupby(["Horizon (s)", "Dataset"]).agg(
            mean_change_pct=("change", "mean"), std_change_pct=("change", "std"), splits_won=("wins", "sum"))
        tabla["Model"], tabla["Output"] = mod, salida
        filas.append(tabla.reset_index())
comparacion = pd.concat(filas, ignore_index=True)
comparacion["splits_won"] = comparacion["splits_won"].astype(int).astype(str) + f"/{N_PARTICIONES}"
display(comparacion.round(2))

# Save the tables for the article
resultados.to_csv("results_all_splits.csv", index=False)
resumen.to_csv("results_summary_mean_std.csv")
comparacion.to_csv("results_vs_persistence.csv", index=False)

## 10. Figures for the article

All figures are generated from the computed predictions (no values typed by hand) and saved as PDF and 300-dpi PNG in `figures/`.

In [ ]:
# Plotting setup.
import matplotlib.pyplot as plt

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

def guardar_figura(fig, nombre):
    fig.savefig(os.path.join(FIG_DIR, f"{nombre}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{nombre}.png"), dpi=300, bbox_inches="tight")
    print(f"Saved: {FIG_DIR}/{nombre}.pdf and .png")

D = datos_por_h[HORIZONTE_PRINCIPAL]   # first split (figures 6-8)

In [ ]:
# Figure 6: training and validation loss (main horizon, model with the physics term).
h = D["historiales"]["CNN-LSTM + PIDL"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["loss"], label="Training loss")
ax.plot(h["val_loss"], label="Validation loss")
mejor = int(np.argmin(h["val_loss"]))
ax.axvline(mejor, color="gray", ls=":", label=f"Best epoch ({mejor + 1})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Total loss (standardized units)")
ax.set_title("CNN-LSTM + PIDL training (1.0 s horizon)")
ax.legend()
guardar_figura(fig, "fig6_training_loss")
plt.show()

In [ ]:
# Figure 7: predicted vs. actual values (main horizon, model with the physics term).
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for fila, ds in enumerate(["PandaSet", "KITTI"]):
    X, g, v, m = D["conjuntos"][ds]
    pg, pv = D["predicciones"][("CNN-LSTM + PIDL", ds)]
    for col, (real, pred, unidad, nombre) in enumerate([(g, pg, "heading rate (deg/frame)", "heading rate"),
                                                         (v, pv, "speed (km/h)", "speed")]):
        ax = axes[fila, col]
        ax.scatter(real, pred, s=6, alpha=0.4)
        lim = [min(real.min(), pred.min()), max(real.max(), pred.max())]
        ax.plot(lim, lim, "k--", lw=1, label="Ideal (y = x)")
        etiqueta = "PandaSet (test sequences)" if ds == "PandaSet" else "KITTI Road (zero-shot)"
        ax.set_title(f"{etiqueta} - {nombre}\n$R^2$ = {r2_score(real, pred):.3f}")
        ax.set_xlabel(f"Actual {unidad}")
        ax.set_ylabel(f"Predicted {unidad}")
        ax.legend(loc="upper left")
fig.tight_layout()
guardar_figura(fig, "fig7_pred_vs_actual")
plt.show()

In [ ]:
# Figure 8: speed along the first KITTI sequence (main horizon).
primera = ids_kitti[0]
n1 = len(construir_secuencias(dfs_kitti[primera], HORIZONTE_PRINCIPAL)[0])
X, g, v, m = D["conjuntos"]["KITTI"]
_, pv = D["predicciones"][("CNN-LSTM + PIDL", "KITTI")]
_, pp = D["predicciones"][("Persistence baseline", "KITTI")]
t = np.arange(n1) / FRECUENCIA_HZ

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, v[:n1], label="Actual mean speed over the next 1.0 s (OXTS)")
ax.plot(t, pv[:n1], label="CNN-LSTM + PIDL")
ax.plot(t, pp[:n1], ":", label="Persistence baseline")
y_sup = 1.3 * max(v[:n1].max(), pv[:n1].max())
dentro = m[:n1] < y_sup
if dentro.any():
    ax.plot(t, np.where(dentro, m[:n1], np.nan), "--", label="Physical $v_{max}$")
else:
    ax.plot([], [], "--", label="Physical $v_{max}$ (above plotted range)")
ax.set_ylim(0, y_sup)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Speed (km/h)")
ax.set_title(f"KITTI {primera} - zero-shot speed prediction")
ax.legend(loc="lower right")
guardar_figura(fig, "fig8_kitti_speed")
plt.show()

In [ ]:
# Figure 9: relative MSE change of the CNN-LSTM + PIDL with respect to the persistence baseline
# (mean +/- std over the splits; labels show in how many splits the network has lower MSE).
comp = comparacion[comparacion["Model"] == "CNN-LSTM + PIDL"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, salida, nombre in zip(axes, ["heading rate", "speed"], ["Heading rate", "Speed"]):
    for ds, mk, off in [("PandaSet", "-o", -0.04), ("KITTI", "--s", 0.04)]:
        sub = comp[(comp["Output"] == salida) & (comp["Dataset"] == ds)].sort_values("Horizon (s)")
        ax.errorbar(sub["Horizon (s)"] + off, sub["mean_change_pct"], yerr=sub["std_change_pct"],
                    fmt=mk, capsize=3, label="PandaSet (test)" if ds == "PandaSet" else "KITTI (zero-shot)")
        for x, y, w in zip(sub["Horizon (s)"] + off, sub["mean_change_pct"], sub["splits_won"]):
            ax.annotate(w, (x, y), textcoords="offset points", xytext=(8, -3), fontsize=8.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks([h / FRECUENCIA_HZ for h in HORIZONTES])
    ax.set_xlabel("Prediction horizon (s)")
    ax.set_ylabel("MSE change vs. persistence (%)")
    ax.set_title(f"{nombre} (negative = lower error than persistence)", fontsize=10.5)
axes[1].legend(loc="upper left")
fig.tight_layout()
guardar_figura(fig, "fig9_mse_vs_persistence")
plt.show()

In [ ]:
# Download all figures and the results table in a single zip file.
import shutil
from google.colab import files
for f in ["results_all_splits.csv", "results_summary_mean_std.csv", "results_vs_persistence.csv"]:
    shutil.copy(f, FIG_DIR)
shutil.make_archive("figures", "zip", FIG_DIR)
files.download("figures.zip")